# Day 9: Knockout Match Prediction Logic

In [3]:
import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path

## Loading Data

In [4]:
def find_project_root():
    current = Path.cwd()

    for path in [current] + list(current.parents):
        if (path / "data").exists() and (path / "models").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root. Make sure this notebook is inside the project folder."
    )


PROJECT_ROOT = find_project_root()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", DATA_PROCESSED)
print("Models folder:", MODELS_DIR)

Project root: /Users/rayhanrinzan/fifa-world-cup-predictor
Processed data folder: /Users/rayhanrinzan/fifa-world-cup-predictor/data/processed
Models folder: /Users/rayhanrinzan/fifa-world-cup-predictor/models


## Loading Model and Metadata

In [5]:
model_path = MODELS_DIR / "current_best_model_with_elo.pkl"
metadata_path = MODELS_DIR / "current_best_model_with_elo_metadata.json"

model = joblib.load(model_path)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

features = metadata["features"]

print("Loaded model:", metadata["model_name"])
print("Features used by model:")
features

Loaded model: gradient_boosting_balanced_clean_elo
Features used by model:


['last_5_points_per_match_diff',
 'last_5_goals_for_per_match_diff',
 'last_5_goals_against_per_match_diff',
 'last_5_goal_difference_per_match_diff',
 'last_5_win_diff',
 'last_5_draw_diff',
 'last_5_loss_diff',
 'elo_diff']

## Load Team Form and Elo Data

In [6]:
team_matches = pd.read_csv(DATA_PROCESSED / "team_matches.csv")
final_elo_df = pd.read_csv(DATA_PROCESSED / "final_elo_ratings.csv")

team_matches["date"] = pd.to_datetime(team_matches["date"])

team_matches_played = team_matches.dropna(
    subset=["goals_for", "goals_against"]
).copy()

team_matches_played = team_matches_played.sort_values(["team", "date"])

final_elo_map = dict(zip(final_elo_df["team"], final_elo_df["final_elo"]))

print("Team matches:", team_matches_played.shape)
print("Teams with Elo ratings:", len(final_elo_map))

team_matches_played.head()

Team matches: (98756, 13)
Teams with Elo ratings: 336


,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home,result,points,goal_difference
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,Sukhumi,Georgia,False,True,draw,1,0.0
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,Stepanakert,Azerbaijan,False,False,loss,0,-3.0
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,Sukhumi,Georgia,False,True,win,3,3.0
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,Östersund,Sweden,True,True,draw,1,0.0
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,Östersund,Sweden,False,False,win,3,1.0


In [7]:
all_teams = sorted(
    set(team_matches_played["team"].dropna()) | set(final_elo_df["team"].dropna())
)

team_lookup = {team.lower(): team for team in all_teams}


def resolve_team_name(team_name):
    """
    Resolves team names while allowing different capitalization.

    Example:
    'france' -> 'France'
    """

    if team_name in all_teams:
        return team_name

    team_name_lower = team_name.lower()

    if team_name_lower in team_lookup:
        return team_lookup[team_name_lower]

    possible_matches = [
        team for team in all_teams
        if team_name_lower in team.lower()
    ]

    if len(possible_matches) > 0:
        raise ValueError(
            f"Team '{team_name}' was not found exactly. Did you mean one of these? {possible_matches[:10]}"
        )

    raise ValueError(f"Team '{team_name}' was not found in the dataset.")

## Create Team Form, Build Match Feature, and Prediction Functions

In [9]:
def get_current_team_form(team_name, n=5):
    """
    Gets a team's most recent form from its last n played matches.
    """

    team_name = resolve_team_name(team_name)

    team_history = (
        team_matches_played[team_matches_played["team"] == team_name]
        .sort_values("date")
        .tail(n)
    )

    matches_used = len(team_history)

    if matches_used == 0:
        return {
            "last_5_points_per_match": 0,
            "last_5_goals_for_per_match": 0,
            "last_5_goals_against_per_match": 0,
            "last_5_goal_difference_per_match": 0,
            "last_5_win": 0,
            "last_5_draw": 0,
            "last_5_loss": 0,
            "matches_used": 0
        }

    return {
        "last_5_points_per_match": team_history["points"].sum() / matches_used,
        "last_5_goals_for_per_match": team_history["goals_for"].sum() / matches_used,
        "last_5_goals_against_per_match": team_history["goals_against"].sum() / matches_used,
        "last_5_goal_difference_per_match": team_history["goal_difference"].sum() / matches_used,
        "last_5_win": (team_history["result"] == "win").sum(),
        "last_5_draw": (team_history["result"] == "draw").sum(),
        "last_5_loss": (team_history["result"] == "loss").sum(),
        "matches_used": matches_used
    }

In [10]:
form_features = [
    "last_5_points_per_match",
    "last_5_goals_for_per_match",
    "last_5_goals_against_per_match",
    "last_5_goal_difference_per_match",
    "last_5_win",
    "last_5_draw",
    "last_5_loss"
]


def build_match_features(team_a, team_b):
    """
    Builds one feature row for Team A vs Team B using the same feature columns
    that the saved model expects.
    """

    team_a = resolve_team_name(team_a)
    team_b = resolve_team_name(team_b)

    team_a_form = get_current_team_form(team_a)
    team_b_form = get_current_team_form(team_b)

    row = {}

    for feature in form_features:
        diff_col = f"{feature}_diff"
        row[diff_col] = team_a_form[feature] - team_b_form[feature]

    team_a_elo = final_elo_map.get(team_a, 1500)
    team_b_elo = final_elo_map.get(team_b, 1500)

    row["elo_diff"] = team_a_elo - team_b_elo

    # Make sure every model feature exists.
    # Any missing expected feature gets neutral value 0.
    for feature in features:
        if feature not in row:
            row[feature] = 0

    X_match = pd.DataFrame([row])[features]

    return X_match

In [11]:
def predict_match(team_a, team_b):
    """
    Predicts a normal match outcome from Team A's perspective.
    """

    team_a = resolve_team_name(team_a)
    team_b = resolve_team_name(team_b)

    X_match = build_match_features(team_a, team_b)

    predicted_label = model.predict(X_match)[0]
    probabilities = model.predict_proba(X_match)[0]

    probability_map = dict(zip(model.classes_, probabilities))

    team_a_win_prob = probability_map.get("win", 0)
    draw_prob = probability_map.get("draw", 0)
    team_b_win_prob = probability_map.get("loss", 0)

    result = pd.DataFrame([
        {
            "team_a": team_a,
            "team_b": team_b,
            "predicted_outcome_from_team_a_perspective": predicted_label,
            "team_a_win_probability": team_a_win_prob,
            "draw_probability": draw_prob,
            "team_b_win_probability": team_b_win_prob,
        }
    ])

    return result

## Create Function to Predict Knockouts

In [12]:
def predict_knockout_match(team_a, team_b):
    """
    Predicts which team advances in a knockout match.

    Beginner assumption:
    If the match is a draw after normal time, both teams get half of the draw probability.
    """

    prediction = predict_match(team_a, team_b)

    team_a = prediction.loc[0, "team_a"]
    team_b = prediction.loc[0, "team_b"]

    team_a_win_prob = prediction.loc[0, "team_a_win_probability"]
    draw_prob = prediction.loc[0, "draw_probability"]
    team_b_win_prob = prediction.loc[0, "team_b_win_probability"]

    team_a_adv_prob = team_a_win_prob + 0.5 * draw_prob
    team_b_adv_prob = team_b_win_prob + 0.5 * draw_prob

    predicted_advancing_team = (
        team_a if team_a_adv_prob >= team_b_adv_prob else team_b
    )

    prediction["team_a_advancement_probability"] = team_a_adv_prob
    prediction["team_b_advancement_probability"] = team_b_adv_prob
    prediction["predicted_advancing_team"] = predicted_advancing_team

    return prediction

In [13]:
def display_knockout_prediction(team_a, team_b):
    prediction = predict_knockout_match(team_a, team_b)

    row = prediction.iloc[0]

    print(f"{row['team_a']} vs {row['team_b']}")
    print("-" * 50)
    print(f"{row['team_a']} win probability: {row['team_a_win_probability']:.3f}")
    print(f"Draw probability: {row['draw_probability']:.3f}")
    print(f"{row['team_b']} win probability: {row['team_b_win_probability']:.3f}")
    print()
    print(f"{row['team_a']} advancement probability: {row['team_a_advancement_probability']:.3f}")
    print(f"{row['team_b']} advancement probability: {row['team_b_advancement_probability']:.3f}")
    print()
    print(f"Predicted advancing team: {row['predicted_advancing_team']}")

    return prediction

In [14]:
display_knockout_prediction("France", "Brazil")

France vs Brazil
--------------------------------------------------
France win probability: 0.336
Draw probability: 0.395
Brazil win probability: 0.269

France advancement probability: 0.533
Brazil advancement probability: 0.467

Predicted advancing team: France


,team_a,team_b,predicted_outcome_from_team_a_perspective,team_a_win_probability,draw_probability,team_b_win_probability,team_a_advancement_probability,team_b_advancement_probability,predicted_advancing_team
0,France,Brazil,draw,0.336211,0.394566,0.269223,0.533494,0.466506,France


## Running some tests

In [15]:
test_matchups = [
    ("France", "Brazil"),
    ("Argentina", "Germany"),
    ("England", "Spain"),
    ("Portugal", "Netherlands"),
    ("United States", "Mexico"),
    ("Japan", "South Korea")
]

knockout_predictions = []

for team_a, team_b in test_matchups:
    prediction = predict_knockout_match(team_a, team_b)
    knockout_predictions.append(prediction)

knockout_predictions_df = pd.concat(knockout_predictions, ignore_index=True)

knockout_predictions_df

,team_a,team_b,predicted_outcome_from_team_a_perspective,team_a_win_probability,draw_probability,team_b_win_probability,team_a_advancement_probability,team_b_advancement_probability,predicted_advancing_team
0,France,Brazil,draw,0.336211,0.394566,0.269223,0.533494,0.466506,France
1,Argentina,Germany,win,0.445704,0.382240,0.172056,0.636824,0.363176,Argentina
2,England,Spain,loss,0.212127,0.386437,0.401436,0.405346,0.594654,Spain
3,Portugal,Netherlands,win,0.378319,0.374693,0.246989,0.565665,0.434335,Portugal
4,United States,Mexico,loss,0.278302,0.328289,0.393409,0.442447,0.557553,Mexico
5,Japan,South Korea,win,0.437537,0.376273,0.186190,0.625673,0.374327,Japan


In [16]:
knockout_predictions_df["max_advancement_probability"] = knockout_predictions_df[
    [
        "team_a_advancement_probability",
        "team_b_advancement_probability"
    ]
].max(axis=1)

knockout_predictions_df.sort_values(
    "max_advancement_probability",
    ascending=False
)

,team_a,team_b,predicted_outcome_from_team_a_perspective,team_a_win_probability,draw_probability,team_b_win_probability,team_a_advancement_probability,team_b_advancement_probability,predicted_advancing_team,max_advancement_probability
1,Argentina,Germany,win,0.445704,0.382240,0.172056,0.636824,0.363176,Argentina,0.636824
5,Japan,South Korea,win,0.437537,0.376273,0.186190,0.625673,0.374327,Japan,0.625673
2,England,Spain,loss,0.212127,0.386437,0.401436,0.405346,0.594654,Spain,0.594654
3,Portugal,Netherlands,win,0.378319,0.374693,0.246989,0.565665,0.434335,Portugal,0.565665
4,United States,Mexico,loss,0.278302,0.328289,0.393409,0.442447,0.557553,Mexico,0.557553
0,France,Brazil,draw,0.336211,0.394566,0.269223,0.533494,0.466506,France,0.533494


## Saving Samples

In [17]:
output_path = DATA_PROCESSED / "example_knockout_predictions.csv"

knockout_predictions_df.to_csv(output_path, index=False)

print("Saved example knockout predictions to:", output_path)

Saved example knockout predictions to: /Users/rayhanrinzan/fifa-world-cup-predictor/data/processed/example_knockout_predictions.csv


## Conclusion

This notebook created the first knockout-stage prediction layer.

The saved model still predicts normal match outcomes:

- Team A win
- Draw
- Team A loss

For knockout-stage use, the draw probability is converted into advancement probability using:

Team A advances = Team A win probability + 0.5 × draw probability  
Team B advances = Team B win probability + 0.5 × draw probability

This is a simple starting assumption. It lets us move forward into bracket simulation while keeping the logic easy to understand.

The next notebook will build a bracket simulator that repeatedly calls `predict_knockout_match()` to advance teams through rounds.